# NeuroXVocal — Full Training Pipeline on Google Colab

**Dataset used:** ADReSSo — only the `diagnosis` subset (AD vs CN binary classification).
- Training: `diagnosis-train/diagnosis/train/audio/ad/` and `cn/`
- Testing:  `diagnosis-test/diagnosis/test-dist/audio/`
- Segmentation CSVs and the progression subset are NOT used.

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## 0. Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime -> Change runtime type -> T4 GPU')

## 1. Install Dependencies

In [ ]:
%%capture
!pip install openai-whisper librosa praat-parselmouth soundfile torchaudio
!pip install transformers==4.45.2 sentencepiece
!pip install scikit-learn pandas numpy tqdm joblib

## 2. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

if not os.path.exists('/content/NeuroXVocal'):
    !git clone https://github.com/homiyano/NeuroXVocal.git /content/NeuroXVocal
else:
    print('Repo already cloned')

%cd /content/NeuroXVocal
!ls

## 3. Dataset Paths

Upload your ADReSSo dataset to Google Drive first, then set the root path below.

Expected Drive layout (just upload the zip/folder as-is from ADReSSo):
```
MyDrive/ADReSSo/
    diagnosis-train/diagnosis/train/audio/
        ad/   <- adrso024.wav, adrso025.wav ...
        cn/   <- adrso002.wav, adrso003.wav ...
    diagnosis-test/diagnosis/test-dist/audio/
        adrsdt1.wav, adrsdt2.wav ...
```

In [ ]:
# ── SET THIS TO WHERE YOU UPLOADED ADReSSo ──────────────────────────
ADRESSO_ROOT = '/content/drive/MyDrive/ADReSSo'
# ────────────────────────────────────────────────────────────────────

AD_WAV_DIR   = os.path.join(ADRESSO_ROOT, 'diagnosis-train/diagnosis/train/audio/ad')
CN_WAV_DIR   = os.path.join(ADRESSO_ROOT, 'diagnosis-train/diagnosis/train/audio/cn')
TEST_WAV_DIR = os.path.join(ADRESSO_ROOT, 'diagnosis-test/diagnosis/test-dist/audio')

# Verify paths exist
for label, path in [('AD train', AD_WAV_DIR), ('CN train', CN_WAV_DIR), ('Test', TEST_WAV_DIR)]:
    exists = os.path.isdir(path)
    wavs   = len([f for f in os.listdir(path) if f.endswith('.wav')]) if exists else 0
    print(f'{label}: {"OK" if exists else "NOT FOUND"}  ({wavs} .wav files)  {path}')

# Local working dir (Colab disk, faster than Drive for I/O)
WORK_DIR = '/content/processed_data'
AD_WORK  = os.path.join(WORK_DIR, 'ad')
CN_WORK  = os.path.join(WORK_DIR, 'cn')
os.makedirs(AD_WORK, exist_ok=True)
os.makedirs(CN_WORK, exist_ok=True)

## 4. Step I — Data Extraction

### 4a. Transcribe Audio with Whisper

In [ ]:
import whisper
from pathlib import Path

def transcribe_dir(wav_dir, out_dir, model):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    wavs = sorted(Path(wav_dir).glob('*.wav'))
    print(f'  Found {len(wavs)} files in {wav_dir}')
    for wav in wavs:
        txt_file = Path(out_dir) / (wav.stem + '.txt')
        if txt_file.exists():
            continue  # skip already done
        result = model.transcribe(str(wav))
        txt_file.write_text(result['text'], encoding='utf-8')
        print(f'  Transcribed {wav.name}')
    print(f'  Done -> {out_dir}')

whisper_model = whisper.load_model('base')

print('Transcribing AD...')
transcribe_dir(AD_WAV_DIR, AD_WORK, whisper_model)

print('Transcribing CN...')
transcribe_dir(CN_WAV_DIR, CN_WORK, whisper_model)

### 4b. Extract Audio Features (librosa + parselmouth)

In [ ]:
AD_FEATURES_RAW = os.path.join(AD_WORK, 'audio_features_ad.csv')
CN_FEATURES_RAW = os.path.join(CN_WORK, 'audio_features_cn.csv')

!python /content/NeuroXVocal/src/data_extraction/extract_audio_features.py \
    {AD_WAV_DIR} --output_csv {AD_FEATURES_RAW}

!python /content/NeuroXVocal/src/data_extraction/extract_audio_features.py \
    {CN_WAV_DIR} --output_csv {CN_FEATURES_RAW}

### 4c. Extract Audio Embeddings (Wav2Vec2)

In [ ]:
AD_EMB_RAW = os.path.join(AD_WORK, 'audio_embeddings_ad.csv')
CN_EMB_RAW = os.path.join(CN_WORK, 'audio_embeddings_cn.csv')

!python /content/NeuroXVocal/src/data_extraction/extract_audio_embeddings.py \
    {AD_WAV_DIR} --output_csv {AD_EMB_RAW}

!python /content/NeuroXVocal/src/data_extraction/extract_audio_embeddings.py \
    {CN_WAV_DIR} --output_csv {CN_EMB_RAW}

## 5. Step II — Preprocessing

### 5a. Preprocess Transcriptions

In [ ]:
AD_TEXT_PROC = os.path.join(WORK_DIR, 'ad_text_processed')
CN_TEXT_PROC = os.path.join(WORK_DIR, 'cn_text_processed')

!python /content/NeuroXVocal/src/data_processing/preprocess_texts.py \
    {AD_WORK} {AD_TEXT_PROC}

!python /content/NeuroXVocal/src/data_processing/preprocess_texts.py \
    {CN_WORK} {CN_TEXT_PROC}

### 5b. Preprocess Audio Features

In [ ]:
import shutil, os

SCALER_FEATURES = '/content/NeuroXVocal/src/inference/scaler_params_audio_features.pkl'

AD_FEAT_PROC_DIR = os.path.join(WORK_DIR, 'ad_features_processed')
CN_FEAT_PROC_DIR = os.path.join(WORK_DIR, 'cn_features_processed')
os.makedirs(AD_FEAT_PROC_DIR, exist_ok=True)
os.makedirs(CN_FEAT_PROC_DIR, exist_ok=True)

!python /content/NeuroXVocal/src/data_processing/preprocess_audio_features.py \
    --input_path {AD_FEATURES_RAW} \
    --output_path {AD_FEAT_PROC_DIR} \
    --scaler_path {SCALER_FEATURES}

!python /content/NeuroXVocal/src/data_processing/preprocess_audio_features.py \
    --input_path {CN_FEATURES_RAW} \
    --output_path {CN_FEAT_PROC_DIR} \
    --scaler_path {SCALER_FEATURES}

AD_FEAT_FINAL = os.path.join(AD_WORK, 'audio_features_ad_processed.csv')
CN_FEAT_FINAL = os.path.join(CN_WORK, 'audio_features_cn_processed.csv')
shutil.copy(os.path.join(AD_FEAT_PROC_DIR, 'audio_features_ad.csv'), AD_FEAT_FINAL)
shutil.copy(os.path.join(CN_FEAT_PROC_DIR, 'audio_features_cn.csv'), CN_FEAT_FINAL)
print('Audio features preprocessed.')

### 5c. Preprocess Audio Embeddings

In [ ]:
import pandas as pd, numpy as np, joblib

SCALER_EMB = '/content/NeuroXVocal/src/inference/scaler_params_audio_emb.pkl'

def preprocess_embeddings(input_csv, output_csv, scaler_path):
    df = pd.read_csv(input_csv)
    patient_ids = df['patient_id'].values
    features = df.drop(columns=['patient_id'])
    features = features.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaler = joblib.load(scaler_path)
    scaled = scaler.transform(features)
    df_out = pd.DataFrame(scaled, columns=features.columns)
    df_out.insert(0, 'patient_id', patient_ids)
    df_out.to_csv(output_csv, index=False)
    print(f'Saved {output_csv}')

AD_EMB_FINAL = os.path.join(AD_WORK, 'audio_embeddings_ad_processed.csv')
CN_EMB_FINAL = os.path.join(CN_WORK, 'audio_embeddings_cn_processed.csv')

preprocess_embeddings(AD_EMB_RAW, AD_EMB_FINAL, SCALER_EMB)
preprocess_embeddings(CN_EMB_RAW, CN_EMB_FINAL, SCALER_EMB)

## 6. Verify Everything is Ready

In [ ]:
from pathlib import Path

checks = [
    ('AD features',   AD_FEAT_FINAL),
    ('CN features',   CN_FEAT_FINAL),
    ('AD embeddings', AD_EMB_FINAL),
    ('CN embeddings', CN_EMB_FINAL),
]
for name, path in checks:
    df = pd.read_csv(path)
    print(f'{name}: {df.shape[0]} patients, {df.shape[1]-1} features')

# Check all feature patients have matching transcripts
ad_feat_ids = set(pd.read_csv(AD_FEAT_FINAL)['patient_id'].astype(str))
cn_feat_ids = set(pd.read_csv(CN_FEAT_FINAL)['patient_id'].astype(str))
ad_txt_ids  = {p.stem for p in Path(AD_TEXT_PROC).glob('*.txt')}
cn_txt_ids  = {p.stem for p in Path(CN_TEXT_PROC).glob('*.txt')}

ad_miss = ad_feat_ids - ad_txt_ids
cn_miss = cn_feat_ids - cn_txt_ids
if ad_miss: print(f'WARNING: {len(ad_miss)} AD patients missing transcripts:', ad_miss)
if cn_miss: print(f'WARNING: {len(cn_miss)} CN patients missing transcripts:', cn_miss)
if not ad_miss and not cn_miss:
    print('All patients have transcripts. Ready to train!')

## 7. Configure & Train

In [ ]:
RESULTS_DIR = '/content/NeuroXVocal/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

config = f'''import os

AD_TEXT_DIR        = r"{AD_TEXT_PROC}"
CN_TEXT_DIR        = r"{CN_TEXT_PROC}"
AD_CSV             = r"{AD_FEAT_FINAL}"
CN_CSV             = r"{CN_FEAT_FINAL}"
AD_EMBEDDING_CSV   = r"{AD_EMB_FINAL}"
CN_EMBEDDING_CSV   = r"{CN_EMB_FINAL}"

TEXT_EMBEDDING_MODEL   = 'microsoft/deberta-v3-base'
NUM_MFCC_FEATURES      = 47
NUM_EMBEDDING_FEATURES = 768
AUDIO_CHANNELS         = 1
CUDA                   = True

BATCH_SIZE              = 8
EPOCHS                  = 50
LEARNING_RATE           = 1e-5
WEIGHT_DECAY            = 1e-4
NUM_FOLDS               = 5
SAVE_BEST_MODEL         = True
EARLY_STOPPING_PATIENCE = 10

SAVE_MODEL_PATH = r"{RESULTS_DIR}/model"
LOG_PATH        = r"{RESULTS_DIR}/training.log"
'''

with open('/content/NeuroXVocal/src/train/config.py', 'w') as f:
    f.write(config)
print('Config written. Starting training...')

In [ ]:
# ~2-4 hours on T4. Each fold saves its best checkpoint to results/
%cd /content/NeuroXVocal/src/train
!python main.py

## 8. Save Results to Drive (in case Colab disconnects)

In [ ]:
import glob, shutil

model_files = sorted(glob.glob(f'{RESULTS_DIR}/*.pth'))
print('Saved model checkpoints:')
for m in model_files:
    print(f'  {os.path.basename(m)}  ({os.path.getsize(m)/1e6:.1f} MB)')

DRIVE_RESULTS = '/content/drive/MyDrive/NeuroXVocal_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
for f in glob.glob(f'{RESULTS_DIR}/*'):
    shutil.copy(f, os.path.join(DRIVE_RESULTS, os.path.basename(f)))
    print(f'  Copied {os.path.basename(f)} to Drive')

## 9. Download Best Model to Your Mac

In [ ]:
from google.colab import files
import zipfile

zip_path = '/content/neuroxvocal_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(f'{RESULTS_DIR}/*.pth'):
        zf.write(f, os.path.basename(f))
    log = f'{RESULTS_DIR}/training.log'
    if os.path.exists(log):
        zf.write(log, 'training.log')

print('Downloading zip...')
files.download(zip_path)

## 10. View Training Log

In [ ]:
log_path = f'{RESULTS_DIR}/training.log'
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    # Show last 80 lines
    print(''.join(lines[-80:]))
else:
    print('Log not found yet.')

---
## After Training — Use Locally

1. Unzip the downloaded file — you'll get files like `model_fold1_best.pth` ... `model_fold5_best.pth`
2. Pick the best fold (check `training.log` for lowest val loss)
3. On your Mac:
```bash
mkdir -p "/Users/homi/Documents/Vibe Coding/NeuroXVocal/results"
cp model_fold3_best.pth "/Users/homi/Documents/Vibe Coding/NeuroXVocal/results/best.pth"
```
4. Run the app:
```bash
cd "/Users/homi/Documents/Vibe Coding/NeuroXVocal/app"
../venv/bin/python -m streamlit run neuroxvocal_app.py
```